# 3_1 Gold — Leads Activities Summary

Builds three Gold tables per spec Section 3:

1. **STAFF_DIM** — Deduplicated staff/user dimension with real names/emails
   - Coaches, CSMs, setters/closers, and partner accounts (everyone who can act in Close CRM)
   - Replaces masked user hashes (`user_<43hex>`) with real human names
   - Used by LEADS_ACTIVITIES_SUMMARY for name enrichment and by SALES_DETAILS for setter/closer
   - Dedup: `USER_ID`, latest by `DATE_UPDATED DESC`

2. **LEADS_ACTIVITY_FACT** — Deduplicated activity fact table enriched with real staff names
   - One row per ACTIVITY_ID (latest by `DATE_UPDATED DESC → INSERT_DATE DESC`) — collapses silver's CDC version history into current state
   - Attributed to staff via staff_dim (`created_by` hash → real name/email)

3. **STAFF_ACCOUNT_DIM** — One row per staff PERSON: the full staff account view
   - Identity (3_00 staff_emails): all emails/hashes (48-char logins + masked), lead/contact links, staff flags
   - CRM directory profile (3_01 staff_dim): real names, image, dates — joined via the EMAIL bridge (the two 48-char id spaces are disjoint)
   - Roles with labels kept separate: `n_csm_leads`/`csm_lead_ids`, `n_setter_sales`/`setter_lead_ids`, `n_closer_sales`/`closer_lead_ids`
   - Fanout guard `n_crm_matches` (one person may match several CRM directory rows); shared emails can attribute nothing (unique-email rule)
   - For per-staff analysis downstream (3_15–3_18 queries and any health-by-coach view)

In [0]:
%sql
-- # STAFF_DIM — Deduplicated staff/user dimension with real names/emails
-- Coaches, CSMs, setters/closers, and partner accounts (everyone who can act in Close CRM)
-- Replaces masked user hashes (user_<43hex>) with real human names
-- Used by LEADS_ACTIVITIES_SUMMARY for name enrichment and by SALES_DETAILS for setter/closer
-- Dedup: USER_ID, latest by DATE_UPDATED DESC
CREATE OR REPLACE TABLE crm_ingestion.gold.staff_dim AS (
    SELECT 
        csm_id,
        image,
        email,
        first_name,
        last_name,
        date_created,
        date_updated,
        bronze_insert_date,
        full_name AS csm_name,
        email_normalized AS csm_email
    FROM (
        SELECT
            id AS csm_id,
            image,
            email,
            first_name,
            last_name,
            date_created,
            date_updated,
            bronze_insert_date,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn,
            CONCAT_WS(' ',first_name, last_name) AS full_name,
            LOWER(TRIM(email)) AS email_normalized
        FROM
            crm_ingestion.silver.close_crm_users_processed      
)
    WHERE
        csm_id IS NOT NULL
    AND 
        rn = 1
)


In [0]:
%sql
SELECT 
    *
FROM
    crm_ingestion.gold.staff_dim
LIMIT 10

In [0]:
%sql
-- Validation: how gold.staff_dim (this notebook - the close_crm user directory)
-- relates to gold.staff_emails (3_00 - staff persons from the identity spine).
-- Two candidate join keys: (1) csm_id vs the spine's 48-char login hashes,
-- (2) csm_email vs the person's collected emails.
SELECT
  (SELECT COUNT(*) FROM crm_ingestion.gold.staff_dim)     AS staff_dim_rows,
  (SELECT COUNT(*) FROM crm_ingestion.gold.staff_emails)  AS staff_persons,
  (SELECT MIN(LENGTH(csm_id)) FROM crm_ingestion.gold.staff_dim) AS min_id_len,
  (SELECT MAX(LENGTH(csm_id)) FROM crm_ingestion.gold.staff_dim) AS max_id_len,
  (SELECT COUNT(*) FROM crm_ingestion.gold.staff_dim d
    JOIN crm_ingestion.gold.staff_emails s
      ON array_contains(s.user_32digits, d.csm_id))       AS joined_via_csm_id,
  (SELECT COUNT(*) FROM crm_ingestion.gold.staff_dim d
    JOIN crm_ingestion.gold.staff_emails s
      ON array_contains(s.all_emails, d.csm_email))        AS joined_via_email,
  (SELECT COUNT(*) FROM crm_ingestion.gold.staff_emails s
    WHERE NOT EXISTS (
      SELECT 1 FROM crm_ingestion.gold.staff_dim d
      WHERE array_contains(s.user_32digits, d.csm_id)
         OR array_contains(s.all_emails, d.csm_email)))    AS staff_persons_without_crm_row

In [0]:
%sql
-- # STAFF_ACCOUNT_DIM — One row per staff PERSON (identity spine grain, 189 rows)
-- The full staff account view. Combines:
--   * 3_00 staff_emails  — identity: all hashes (48-char logins + masked), all
--     emails, lead/contact links, classification flags (whole-master rule)
--   * 3_01 staff_dim      — CRM directory profile: real names, image, dates
--   * role attribution    — CSM assignments (lead_dim.csm_id) and sale roles
--
-- BRIDGE: the two 48-char id spaces are DISJOINT (close_crm user ids never
-- appear in the spine's organizer hashes — validated in the cell above), so
-- the ONLY reliable join is EMAIL: staff_dim.csm_email IN staff_emails.all_emails
-- (185/443 directory rows match a person; 6 staff persons have no CRM row).
-- Fanout guard: one person with several real emails can match multiple CRM
-- directory rows — we keep ONE deterministic profile (latest date_updated)
-- and expose n_crm_matches so fanout is visible, never silent.
-- Roles keep their LABELS SEPARATE (setter and closer are distinct columns),
-- attributed at the 48-char id level: sale custom fields -> staff_dim ->
-- UNIQUE email -> person (shared emails cannot attribute). Counts use ALL
-- qualifying sale activities (types 7/8, deduped),
-- not just the latest sale per lead that sales_fact keeps.
CREATE OR REPLACE TABLE crm_ingestion.gold.staff_account_dim AS (
    WITH unique_email_person AS (
        -- EMAIL -> PERSON only when the email is unique across ALL staff persons.
        -- 11 directory emails sit in 2+ staff persons' all_emails (shared
        -- inboxes) - joining through them would credit one sale to ~6 people
        -- (found by the role-attribution validation: 285 closer pairs over only
        -- 43 distinct leads). Shared emails can never attribute a role or a
        -- profile; they stay for LOOKUP only (same principle as 2_8's guards).
        SELECT e.email, MIN(se.master_user_id) AS master_user_id
        FROM crm_ingestion.gold.staff_emails se
        LATERAL VIEW explode(se.all_emails) e AS email
        GROUP BY e.email
        HAVING COUNT(DISTINCT se.master_user_id) = 1
    ),
    profile AS (
        -- Best CRM-directory row per staff person via the UNIQUE email bridge.
        SELECT
            s.master_user_id,
            d.csm_id,
            d.csm_name,
            d.first_name,
            d.last_name,
            d.image,
            d.csm_email,
            d.date_created AS crm_date_created,
            d.date_updated  AS crm_date_updated
        FROM crm_ingestion.gold.staff_emails s
        LEFT JOIN unique_email_person u ON u.master_user_id = s.master_user_id
        LEFT JOIN crm_ingestion.gold.staff_dim d ON d.csm_email = u.email
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY s.master_user_id
            ORDER BY d.date_updated DESC NULLS LAST, d.csm_id NULLS LAST
        ) = 1
    ),
    crm_matches AS (
        -- How many CRM directory rows each person matched (fanout guard).
        SELECT
            se.master_user_id,
            COUNT(DISTINCT d.csm_id) AS n_crm_matches
        FROM crm_ingestion.gold.staff_emails se
        JOIN crm_ingestion.gold.staff_dim d
            ON array_contains(se.all_emails, d.csm_email)
        GROUP BY 1
    ),
    csm_leads AS (
        -- Leads whose assigned CRM user (lead_dim.csm_id) is this person.
        SELECT
            u.master_user_id,
            COUNT(DISTINCT ld.lead_id) AS n_csm_leads,
            COLLECT_SET(ld.lead_id)     AS csm_lead_ids
        FROM crm_ingestion.gold.lead_dim ld
        JOIN crm_ingestion.gold.staff_dim d ON ld.csm_id = d.csm_id
        JOIN unique_email_person u ON u.email = d.csm_email
        GROUP BY 1
    ),
    qualifying_sales AS (
        -- ALL qualifying sale activities (7/8), deduped to current state.
        SELECT
            a.id,
            a.lead_id,
            a.custom_fields['cf_v385AJ8HSgepKQ3rvqo4yOA3nn49eGqz39DOqojJG5M'] AS setter_id,
            a.custom_fields['cf_Lv5lSqLOZwLrNhe5M7kWx2mF8Ge2Z23aw5NUNhbXvVS'] AS closer_id
        FROM crm_ingestion.silver.lead_activites_processed a
        WHERE a.custom_activity_type_id IN (
            'actitype_3E85vFq3a06LlEzXT2N1kS',  -- 7) New Sale
            'actitype_0FNk72Q8eSYX2MVd4A2UFx'   -- 8) New Sale [Custom Payment Plan]
        )
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY a.id ORDER BY a.date_updated DESC, a.bronze_insert_date DESC
        ) = 1
    ),
    setter_sales AS (
        SELECT
            u.master_user_id,
            COUNT(DISTINCT q.lead_id) AS n_setter_sales,
            COLLECT_SET(q.lead_id)    AS setter_lead_ids
        FROM qualifying_sales q
        JOIN crm_ingestion.gold.staff_dim d ON q.setter_id = d.csm_id
        JOIN unique_email_person u ON u.email = d.csm_email
        WHERE q.setter_id IS NOT NULL
        GROUP BY 1
    ),
    closer_sales AS (
        SELECT
            u.master_user_id,
            COUNT(DISTINCT q.lead_id) AS n_closer_sales,
            COLLECT_SET(q.lead_id)    AS closer_lead_ids
        FROM qualifying_sales q
        JOIN crm_ingestion.gold.staff_dim d ON q.closer_id = d.csm_id
        JOIN unique_email_person u ON u.email = d.csm_email
        WHERE q.closer_id IS NOT NULL
        GROUP BY 1
    )
    SELECT
        -- identity (from 3_00 / the spine)
        se.master_user_id,
        se.user_32digits,
        se.real_emails,
        se.masked_emails,
        se.all_emails,
        se.lead_ids,
        se.cont_ids,
        se.source_tables,
        se.is_organizer_login,
        se.is_dea_domain,
        se.is_registered,
        -- CRM directory profile (from 3_01; NULL for the 6 staff with no CRM row)
        p.csm_id,
        p.csm_name,
        p.first_name,
        p.last_name,
        p.image,
        p.csm_email,
        p.crm_date_created,
        p.crm_date_updated,
        COALESCE(cm.n_crm_matches, 0) AS n_crm_matches,
        -- roles, LABELS KEPT SEPARATE
        COALESCE(cs.n_csm_leads, 0)   AS n_csm_leads,
        cs.csm_lead_ids,
        COALESCE(ss.n_setter_sales, 0) AS n_setter_sales,
        ss.setter_lead_ids,
        COALESCE(cl.n_closer_sales, 0) AS n_closer_sales,
        cl.closer_lead_ids
    FROM crm_ingestion.gold.staff_emails se
    LEFT JOIN profile       p  ON p.master_user_id  = se.master_user_id
    LEFT JOIN crm_matches   cm ON cm.master_user_id = se.master_user_id
    LEFT JOIN csm_leads     cs ON cs.master_user_id = se.master_user_id
    LEFT JOIN setter_sales  ss ON ss.master_user_id = se.master_user_id
    LEFT JOIN closer_sales  cl ON cl.master_user_id = se.master_user_id
)

In [0]:
%sql
-- Validation: coverage and role attribution of the combined staff account dim.
-- Expected: 189 person rows, 183 with a CRM profile (6 staff have no CRM row),
-- role totals attributable only via the email bridge.
SELECT
    COUNT(*) AS staff_persons,
    SUM(CASE WHEN csm_name IS NOT NULL THEN 1 ELSE 0 END) AS with_crm_profile,
    SUM(CASE WHEN n_crm_matches > 1 THEN 1 ELSE 0 END)   AS fanout_persons,
    SUM(n_csm_leads)    AS total_csm_lead_assignments,
    SUM(n_setter_sales) AS total_setter_sales,
    SUM(n_closer_sales) AS total_closer_sales,
    MAX(n_csm_leads)    AS max_csm_leads_one_person,
    MAX(n_setter_sales) AS max_setter_sales_one_person,
    MAX(n_closer_sales) AS max_closer_sales_one_person
FROM crm_ingestion.gold.staff_account_dim

In [0]:
%sql
-- Validation: decompose role attribution. person_lead_pairs vs distinct_leads
-- separates multi-role leads (same lead, different setter/closer persons across
-- its sale history - legitimate) from double-person attribution (one directory
-- email shared by several staff persons). This diagnostic deliberately uses the
-- UNGUARDED any-email join: the shared_email row lists the directory emails
-- resolving to 2+ staff persons - exactly the ambiguity the build's
-- unique_email_person guard now excludes (roles and profiles attach only via
-- unique emails; the 11 shared emails stay for lookup only).
WITH qs AS (
    SELECT
        a.id, a.lead_id,
        a.custom_fields['cf_v385AJ8HSgepKQ3rvqo4yOA3nn49eGqz39DOqojJG5M'] AS setter_id,
        a.custom_fields['cf_Lv5lSqLOZwLrNhe5M7kWx2mF8Ge2Z23aw5NUNhbXvVS'] AS closer_id
    FROM crm_ingestion.silver.lead_activites_processed a
    WHERE a.custom_activity_type_id IN (
        'actitype_3E85vFq3a06LlEzXT2N1kS','actitype_0FNk72Q8eSYX2MVd4A2UFx')
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY a.id ORDER BY a.date_updated DESC, a.bronze_insert_date DESC) = 1
),
pairs AS (
    SELECT DISTINCT q.lead_id, se.master_user_id, 'setter' AS role
    FROM qs q
    JOIN crm_ingestion.gold.staff_dim d ON q.setter_id = d.csm_id
    JOIN crm_ingestion.gold.staff_emails se ON array_contains(se.all_emails, d.csm_email)
    WHERE q.setter_id IS NOT NULL
    UNION ALL
    SELECT DISTINCT q.lead_id, se.master_user_id, 'closer'
    FROM qs q
    JOIN crm_ingestion.gold.staff_dim d ON q.closer_id = d.csm_id
    JOIN crm_ingestion.gold.staff_emails se ON array_contains(se.all_emails, d.csm_email)
    WHERE q.closer_id IS NOT NULL
)
SELECT 'sale_roles' AS check, role, COUNT(*) AS person_lead_pairs,
       COUNT(DISTINCT lead_id) AS distinct_leads
FROM pairs GROUP BY role
UNION ALL
SELECT 'directory emails resolving to 2+ staff persons (must be 0)', 'shared_email',
       COUNT(*), NULL
FROM (
    SELECT d.csm_email
    FROM crm_ingestion.gold.staff_dim d
    JOIN crm_ingestion.gold.staff_emails se ON array_contains(se.all_emails, d.csm_email)
    GROUP BY 1 HAVING COUNT(DISTINCT se.master_user_id) > 1
)

In [0]:
%sql
DESCRIBE TABLE crm_ingestion.silver.lead_activites_processed